# DermTriage Demo: Full Inference Pipeline

This notebook demonstrates the complete DermTriage clinical decision support system:

1. **MedSigLIP** classification with MC-Dropout uncertainty
2. **Grad-CAM** visual explanations
3. **MedGemma** natural language clinical assessment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kabirgrover/dermtriage/blob/master/notebooks/03_demo_pipeline.ipynb)

### Setup
Before running, add these secrets in Colab (Settings > Secrets):
- `HF_TOKEN`: Your HuggingFace token (with MedGemma access)
- `KAGGLE_USERNAME` and `KAGGLE_KEY`: For downloading sample images

In [ ]:
# Cell 1: Install dependencies & authenticate
!pip install -q torch torchvision transformers Pillow matplotlib tqdm huggingface-hub gdown kaggle scikit-learn

import os
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
print('Authenticated!')

In [ ]:
# Cell 2: Download checkpoint from Google Drive
import gdown
from pathlib import Path

CHECKPOINT_PATH = Path('/content/best_model.pth')

if not CHECKPOINT_PATH.exists():
    # TODO: Replace with your public Google Drive link
    # gdown.download('https://drive.google.com/uc?id=YOUR_FILE_ID', str(CHECKPOINT_PATH))
    print('Please upload best_model.pth or set the Google Drive link above.')
    print('You can upload manually: click the folder icon on the left, then drag and drop.')
else:
    print(f'Checkpoint found: {CHECKPOINT_PATH}')

In [ ]:
# Cell 3: Download a few sample images from HAM10000
import subprocess, shutil
import pandas as pd
from sklearn.model_selection import train_test_split

SAMPLE_DIR = Path('/content/samples')
DATA_DIR = Path('/content/data')

if not SAMPLE_DIR.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(['kaggle', 'datasets', 'download', '-d', 'kmader/skin-cancer-mnist-ham10000',
                    '-p', str(DATA_DIR), '--unzip'], check=True)
    # Pick a few samples from different classes
    df = pd.read_csv(DATA_DIR / 'HAM10000_metadata.csv')
    img_dirs = [DATA_DIR / 'HAM10000_images_part_1', DATA_DIR / 'HAM10000_images_part_2']
    SAMPLE_DIR.mkdir(exist_ok=True)
    for cls in ['mel', 'nv', 'bcc', 'bkl']:
        rows = df[df['dx'] == cls].head(2)
        for _, row in rows.iterrows():
            img_name = f"{row['image_id']}.jpg"
            for img_dir in img_dirs:
                src = img_dir / img_name
                if src.exists():
                    shutil.copy(src, SAMPLE_DIR / f"{cls}_{img_name}")
                    break
    print(f'Sample images ready: {list(SAMPLE_DIR.glob("*.jpg"))}')
else:
    print(f'Samples already exist: {list(SAMPLE_DIR.glob("*.jpg"))}')

In [ ]:
# Cell 4: Define model and load checkpoint
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import torch.nn.functional as F

CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLASS_INFO = {
    'akiec': {'name': 'Actinic Keratosis', 'risk': 'MODERATE', 'action': 'Dermatology referral within 2-4 weeks'},
    'bcc': {'name': 'Basal Cell Carcinoma', 'risk': 'HIGH', 'action': 'Dermatology referral within 2 weeks'},
    'bkl': {'name': 'Benign Keratosis', 'risk': 'LOW', 'action': 'Routine monitoring'},
    'df': {'name': 'Dermatofibroma', 'risk': 'LOW', 'action': 'No treatment required'},
    'mel': {'name': 'Melanoma', 'risk': 'URGENT', 'action': 'URGENT referral within 48 hours'},
    'nv': {'name': 'Melanocytic Nevus', 'risk': 'LOW', 'action': 'Routine monitoring'},
    'vasc': {'name': 'Vascular Lesion', 'risk': 'LOW', 'action': 'No treatment required'},
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class MedSigLIPClassifier(nn.Module):
    def __init__(self, num_classes=7, dropout_rate=0.3):
        super().__init__()
        from transformers import AutoModel
        full_model = AutoModel.from_pretrained('google/medsiglip-448', torch_dtype=torch.float32)
        self.vision_model = full_model.vision_model
        self.embed_dim = full_model.config.vision_config.hidden_size
        del full_model
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.embed_dim), nn.Dropout(dropout_rate),
            nn.Linear(self.embed_dim, 512), nn.GELU(), nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes)
        )

    def forward(self, pixel_values):
        with torch.no_grad():
            outputs = self.vision_model(pixel_values=pixel_values)
            features = outputs.pooler_output if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None else outputs.last_hidden_state.mean(dim=1)
        return self.classifier(features)


# Load model
print('Loading MedSigLIP classifier...')
model = MedSigLIPClassifier(num_classes=7).to(device)
checkpoint = torch.load(str(CHECKPOINT_PATH), map_location=device, weights_only=False)
if 'classifier_state_dict' in checkpoint:
    model.classifier.load_state_dict(checkpoint['classifier_state_dict'])
else:
    model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Model loaded! (checkpoint epoch: {checkpoint.get("epoch", "?")})')

In [ ]:
# Cell 5: Classify sample images with uncertainty
def preprocess_image(image_path, size=448):
    image = Image.open(image_path).convert('RGB')
    image = image.resize((size, size), Image.BILINEAR)
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    return transform(image).unsqueeze(0), image


def classify_with_uncertainty(model, image_tensor, num_samples=10):
    model.eval()
    image_tensor = image_tensor.to(device)
    with torch.no_grad():
        logits = model(image_tensor)
        probs = torch.softmax(logits, dim=-1)
        confidence, pred_idx = probs.max(dim=-1)

    # MC Dropout uncertainty
    def enable_dropout(m):
        if isinstance(m, nn.Dropout): m.train()
    model.classifier.apply(enable_dropout)
    mc_probs = []
    with torch.no_grad():
        for _ in range(num_samples):
            mc_probs.append(torch.softmax(model(image_tensor), dim=-1))
    mc_probs = torch.stack(mc_probs)
    mean_probs = mc_probs.mean(dim=0)
    entropy = -torch.sum(mean_probs * torch.log(mean_probs + 1e-10), dim=-1).item()
    uncertainty = entropy / np.log(len(CLASS_NAMES))

    pred_class = CLASS_NAMES[pred_idx.item()]
    return {
        'class': pred_class, 'class_name': CLASS_INFO[pred_class]['name'],
        'confidence': confidence.item(), 'uncertainty': uncertainty,
        'risk_level': CLASS_INFO[pred_class]['risk'],
        'action': CLASS_INFO[pred_class]['action'],
        'all_probs': {CLASS_NAMES[i]: probs[0, i].item() for i in range(len(CLASS_NAMES))}
    }


# Run on all sample images
sample_images = sorted(SAMPLE_DIR.glob('*.jpg'))
results = {}

for img_path in sample_images:
    tensor, pil_img = preprocess_image(img_path)
    result = classify_with_uncertainty(model, tensor)
    results[img_path] = (result, pil_img)
    print(f'{img_path.name}: {result["class_name"]} ({result["confidence"]*100:.1f}%, '
          f'risk={result["risk_level"]}, uncertainty={result["uncertainty"]:.3f})')

In [ ]:
# Cell 6: Grad-CAM visualization
class GradCAM:
    def __init__(self, model):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer = model.vision_model.encoder.layers[-1]
        target_layer.register_forward_hook(self._save_act)
        target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, m, i, o):
        self.activations = (o[0] if isinstance(o, tuple) else o).detach()
    def _save_grad(self, m, gi, go):
        self.gradients = (go[0] if isinstance(go, tuple) else go).detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        output = self.model(input_tensor)
        if class_idx is None: class_idx = output.argmax(dim=1).item()
        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0, class_idx] = 1
        output.backward(gradient=one_hot, retain_graph=True)
        weights = self.gradients.mean(dim=1, keepdim=True)
        cam = (weights * self.activations).sum(dim=-1)[:, 1:]
        grid = int(np.sqrt(cam.shape[1]))
        cam = F.relu(cam.reshape(1, grid, grid))
        cam = cam - cam.min()
        if cam.max() > 0: cam = cam / cam.max()
        cam = F.interpolate(cam.unsqueeze(0), size=(input_tensor.shape[2], input_tensor.shape[3]),
                            mode='bilinear', align_corners=False)
        return cam.squeeze().cpu().numpy(), class_idx


# Generate Grad-CAM for each sample
gradcam = GradCAM(model)
n = len(sample_images)
fig, axes = plt.subplots(n, 3, figsize=(15, 5 * n))
if n == 1: axes = axes.reshape(1, -1)

for i, img_path in enumerate(sample_images):
    result, pil_img = results[img_path]
    tensor, _ = preprocess_image(img_path)
    tensor = tensor.to(device)
    heatmap, cidx = gradcam.generate(tensor)

    axes[i, 0].imshow(pil_img)
    axes[i, 0].set_title(f'Original: {img_path.stem}', fontsize=11)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(heatmap, cmap='jet')
    axes[i, 1].set_title('Grad-CAM', fontsize=11)
    axes[i, 1].axis('off')

    img_arr = np.array(pil_img.resize((448, 448)))
    cmap_fn = cm.get_cmap('jet')
    hm_colored = cmap_fn(heatmap)[:, :, :3] * 255
    overlay = np.clip(0.6 * img_arr + 0.4 * hm_colored, 0, 255).astype(np.uint8)
    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title(f'{result["class_name"]} ({result["confidence"]*100:.0f}%)', fontsize=11)
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: MedGemma clinical explanation
from transformers import AutoProcessor, AutoModelForImageTextToText

print('Loading MedGemma for clinical explanation...')
medgemma = AutoModelForImageTextToText.from_pretrained(
    'google/medgemma-4b-it', torch_dtype=torch.bfloat16, device_map='auto')
processor = AutoProcessor.from_pretrained('google/medgemma-4b-it')

# Pick a sample to explain (first melanoma or first sample)
explain_path = sample_images[0]
for p in sample_images:
    if 'mel' in p.name:
        explain_path = p
        break

result, pil_img = results[explain_path]
print(f'\nExplaining: {explain_path.name} -> {result["class_name"]} ({result["confidence"]*100:.1f}%)')

prompt = f"""You are a dermatology AI assistant helping primary care physicians.

Analyze this dermoscopic image. The AI classifier detected: {result['class_name']}
Confidence: {result['confidence']*100:.1f}%
Uncertainty: {'HIGH - recommend expert review' if result['uncertainty'] > 0.3 else 'LOW'}

Provide a brief clinical assessment:
1. Describe key dermoscopic features you observe (2-3 sentences)
2. State whether the classification appears consistent with visual features
3. Note any concerning features that warrant attention

Be concise and clinically focused."""

messages = [{'role': 'user', 'content': [{'type': 'image', 'image': pil_img}, {'type': 'text', 'text': prompt}]}]
inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                        return_dict=True, return_tensors='pt').to(medgemma.device, dtype=torch.bfloat16)
input_len = inputs['input_ids'].shape[-1]

with torch.inference_mode():
    gen = medgemma.generate(**inputs, max_new_tokens=250, do_sample=False)
    gen = gen[0][input_len:]

explanation = processor.decode(gen, skip_special_tokens=True)
print('\n' + '='*60)
print('MedGemma Clinical Assessment:')
print('='*60)
print(explanation)

del medgemma, processor
torch.cuda.empty_cache()

In [ ]:
# Cell 8: Full clinical report
print('\n' + '='*60)
print('DERMTRIAGE CLINICAL DECISION SUPPORT REPORT')
print('='*60)
print(f'Image: {explain_path.name}')
print()
print('-'*60)
print('CLASSIFICATION RESULT')
print('-'*60)
print(f'Diagnosis:   {result["class_name"]}')
print(f'Risk Level:  {result["risk_level"]}')
print(f'Confidence:  {result["confidence"]*100:.1f}%')
print(f'Uncertainty: {result["uncertainty"]:.3f} ({"HIGH" if result["uncertainty"] > 0.3 else "LOW"})')
print()
print('Class Probabilities:')
for cls, prob in sorted(result['all_probs'].items(), key=lambda x: x[1], reverse=True):
    marker = ' <--' if cls == result['class'] else ''
    print(f'  {CLASS_INFO[cls]["name"]:30s}: {prob*100:5.1f}%{marker}')
print()
print('-'*60)
print('RECOMMENDED ACTION')
print('-'*60)
print(result['action'])
print()
print('-'*60)
print('AI CLINICAL ANALYSIS (MedGemma)')
print('-'*60)
print(explanation)
print()
print('='*60)
print('DISCLAIMER: For clinical decision support only.')
print('Final diagnosis requires expert dermatologic evaluation.')
print('='*60)